# From Counts to Argument

**Computational Text Analysis Pack · Module 11 — From Counts to Argument**

*Technique:* Classifying what a number can carry: description, comparison, claim

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/11a_from_counts_to_argument.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

You now have ten methods. The remaining skill is knowing what each result can
carry — and it is the skill that decides whether a paper is accepted.

Every number in this pack is one of three things:

| Kind | Says | Needs |
|---|---|---|
| **Description** | what is in this corpus | the corpus documented |
| **Comparison** | this corpus differs from that one | a named reference, an effect size |
| **Claim** | something about the world | everything above, plus an argument about the sample |

Most published errors are a **description dressed as a claim**.

## The tooling

No new library. This module is about turning numbers into an argument, which is
the part no package does for you.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from collections import Counter

def frequencies(texts):
    """Token counts across a set of documents."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    return counts

def freq_frame(counts, top=25):
    total = sum(counts.values())
    rows = [{"rank": i, "word": w, "count": n, "per_1000": n / total * 1000}
            for i, (w, n) in enumerate(counts.most_common(top), start=1)]
    return pd.DataFrame(rows)

In [ ]:
def log_likelihood(counts_a, counts_b, min_count=10):
    """Words that distinguish corpus A from corpus B.

    The G-squared statistic corpus linguists call *keyness*. A word scores high
    when it is far more common in one corpus than its overall rate predicts.
    Positive `diff` means over-used in A.
    """
    na, nb = sum(counts_a.values()), sum(counts_b.values())
    rows = []
    for w in set(counts_a) | set(counts_b):
        o1, o2 = counts_a.get(w, 0), counts_b.get(w, 0)
        if o1 + o2 < min_count:
            continue
        e1 = na * (o1 + o2) / (na + nb)
        e2 = nb * (o1 + o2) / (na + nb)
        g2 = 2 * ((o1 * np.log(o1 / e1) if o1 else 0)
                  + (o2 * np.log(o2 / e2) if o2 else 0))
        rows.append({"word": w, "count_a": o1, "count_b": o2,
                     "per_1000_a": o1 / na * 1000, "per_1000_b": o2 / nb * 1000,
                     "keyness": g2, "over_used_in": "A" if o1 / na > o2 / nb else "B"})
    out = pd.DataFrame(rows).sort_values("keyness", ascending=False)
    return out.reset_index(drop=True)

In [ ]:
# One result, written three ways. Only one of them is safe.
eng = df[df["language"] == "English"]
health = eng[eng["category"] == "health"]
rest = eng[eng["category"] != "health"]
keys = log_likelihood(frequencies(health["text"]), frequencies(rest["text"]),
                      min_count=25)
top = keys[keys["over_used_in"] == "A"].iloc[0]

w = top["word"]
print(f"the result: '{w}' — {top['per_1000_a']:.2f} per 1,000 in health reporting")
print(f"            vs {top['per_1000_b']:.2f} elsewhere, G² = {top['keyness']:.1f}")
print()
print("DESCRIPTION (safe):")
print(f"  In this corpus, '{w}' occurs at {top['per_1000_a']:.2f} per 1,000 words")
print("  in health-categorised articles.")
print()
print("COMPARISON (safe, with the reference named):")
print(f"  '{w}' is {top['per_1000_a'] / max(top['per_1000_b'], 1e-6):.1f}× more")
print("  frequent in health reporting than in other categories of the same corpus.")
print()
print("CLAIM (needs an argument you have not made):")
print(f"  African journalism frames health through '{w}'.")
print("  ← which African journalism? this is 16 languages, sampled unevenly,")
print("     from whichever outlets MasakhaNEWS could license.")

## The three questions that turn a comparison into a claim

1. **Sample** — is the corpus representative of the thing you are talking about,
   and how do you know?
2. **Mechanism** — is there a reason the difference should exist, beyond the
   difference itself?
3. **Alternative** — what else would produce this pattern?

You cannot answer these with code. They are why the humanities training you
already have is the scarce part of this work, not the Python.

In [ ]:
# Question 3, made concrete: three alternative explanations, testable here.
print("Pattern: health articles over-use a distinctive vocabulary.")
print()
alt = [
    ("The categories were assigned by the corpus builders, not by the outlets",
     "check: how consistent are the labels? read 10 health articles"),
    ("Health articles come from different outlets than political ones",
     "check: the corpus has a url column — compare domains per category"),
    ("The difference is length, not vocabulary",
     "check: compare median n_words per category (02d)"),
]
for i, (claim, test) in enumerate(alt, 1):
    print(f"{i}. {claim}")
    print(f"   {test}")
    print()

In [ ]:
# Run alternative 2. It takes four lines and it is the sort of check that
# separates a defensible paper from a rejected one.
import re
eng = eng.copy()
eng["domain"] = eng["url"].astype(str).str.extract(r"https?://([^/]+)")[0]
ct = pd.crosstab(eng["category"], eng["domain"])
ct = ct.loc[:, ct.sum() >= 20]
if ct.shape[1] > 1:
    pct = ct.div(ct.sum(axis=1), axis=0) * 100
    print("Which outlets each category comes from (%):")
    print(pct.round(0).to_string())
    print()
    print("If the rows differ substantially, your 'category' effect is partly an")
    print("'outlet' effect, and the vocabulary difference may be house style.")
else:
    print("Too few identifiable domains in this sample to run the check —")
    print("which is itself worth reporting as a limit.")

In [ ]:
# Alternative 3, run.
print(eng.groupby("category")["n_words"]
         .agg(docs="size", median="median")
         .query("docs >= 20").to_string(float_format=lambda v: f"{v:,.0f}"))
print()
print("Length differences do not explain a keyness result — keyness is")
print("normalised — but they do explain differences in raw counts, in TTR (02c),")
print("and in anything computed per document.")

### 🔧 Your turn 1

Take a result you produced in an earlier notebook and write it out three ways:
description, comparison, claim.

Then decide which of the three you can actually defend, and delete the other two.

## The evidence ladder, for text

From weakest to strongest, as a reviewer reads it:

1. A frequency count from one corpus
2. A frequency comparison against a named reference
3. …with an effect size and the raw counts reported
4. …that survives a change of reference corpus (04c)
5. …with concordance lines showing the word is used as you claim (05a)
6. …that replicates in an independent corpus

Most DH papers stop at 2. Getting to 4 or 5 costs an afternoon, and this pack has
given you every tool it needs.

In [ ]:
def kwic(texts, term, width=45, limit=12):
    """Keyword in context: every occurrence with the words either side."""
    pattern = re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE)
    rows = []
    for doc_id, text in enumerate(texts):
        s = str(text)
        for m in pattern.finditer(s):
            left = s[max(0, m.start() - width):m.start()].replace("\n", " ")
            right = s[m.end():m.end() + width].replace("\n", " ")
            rows.append({"doc": doc_id, "left": left.rjust(width),
                         "match": m.group(0), "right": right})
            if len(rows) >= limit:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

In [ ]:
# Step 5, in three lines. Do this before you write the sentence.
lines = kwic(health["text"], w, width=42, limit=8)
for _, r in lines.iterrows():
    print(f"{r['left']} [{r['match']}] {r['right']}")
print()
print(f"Now: does '{w}' mean in these lines what your claim assumes it means?")

### 🔧 Your turn 2

Pick your strongest finding and walk it up the ladder as far as you can get today.

Write down which rung you reached and what stopped you. That sentence belongs in
your limits section — a paper that names its own ceiling is much harder to attack
than one that pretends there is none.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The usual outcome is that the description and the comparison
survive and the claim does not — not because the claim is false, but because the
corpus cannot support it alone. That is the normal state of a single-corpus
result. The way to promote it is another corpus, not stronger wording.

**Your turn 2.** Rung 4 (survives a change of reference) and rung 5 (supported by
concordance lines) are both reachable inside this pack with the corpora you
already have. Rung 6 needs data you would have to build. Saying "this finding
survives a change of reference corpus but has not been replicated on an
independent collection" is a precise, honest, and unusually strong claim for a DH
paper.

</details>

## Where this stops

You can tell what a result can carry. Next: **11b**, on the things frequency
cannot show at all.